In [31]:
import pandas as pd

data = pd.read_parquet("train.parquet")
data.head()

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услу...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу ...,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид...,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


## Два протокола локальной валидации Recall@50

**combined**: корпус — объединение уникальных объявлений из train и benchmark_items. Все известные позитивы доступны в корпусе; сохраняем все запросы. Это основная большая проверка.

**benchmark**: корпус — все объявления benchmark_items. Разметка — пересечение train-позитивов с этим корпусом; запросы без доступных позитивов исключаем. Пересекается разметка, а не сам корпус кандидатов: поиск только среди общих item_id искусственно упростил бы задачу.

Используем одно разбиение по нормализованному тексту для обоих протоколов: примерно 70% train, 15% val, 15% test, seed=42. Единица метрики — полная сигнатура из пяти search-полей. Один текст со всеми его локациями/фильтрами относится к одному split.

Val — для подбора параметров; test — после фиксации решения. Все обучаемые модели, статистики и история взаимодействий используют только общие train_pairs. TF-IDF можно обучать на текстах полного корпуса выбранного протокола. Признаки объявления доступны при поиске; скрытая связь «запрос выбрал item» доступна только оценщику.

После замены старых ячеек выполните Restart Kernel → Run All, чтобы не использовать переменные от прежнего протокола.


In [32]:
from pathlib import Path
from collections.abc import Mapping
import pandas as pd
from IPython.display import display

# Задаем константные переменные и считываем данные
DATA_DIR = Path.cwd()
QUERY_COLS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
SPLIT_SEED = 42
SPLITS = ("train", "val", "test")

validation_source = pd.read_parquet(
    DATA_DIR / "train.parquet", columns=QUERY_COLS + ["item_id"]
)
benchmark_corpus = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")
ITEM_COLS = benchmark_corpus.columns.tolist()


# Читаем все признаки объявлений из train и удаляем одинаковые строки.
train_item_catalog = (
    pd.read_parquet(DATA_DIR / "train.parquet", columns=ITEM_COLS)
    .drop_duplicates()
    .reset_index(drop=True)
)

# собираем объединенный датасет
benchmark_item_ids = frozenset(benchmark_corpus["item_id"])
combined_corpus = pd.concat(
    [
        benchmark_corpus,
        train_item_catalog.loc[
            ~train_item_catalog["item_id"].isin(benchmark_item_ids)
        ],
    ],
    ignore_index=True,
)
combined_item_ids = frozenset(combined_corpus["item_id"])
print(f"benchmark: {len(benchmark_corpus):,} объявлений")
print(f"combined:  {len(combined_corpus):,} объявлений")

benchmark: 189,212 объявлений
combined:  515,895 объявлений


In [3]:
combined_corpus.shape

(515895, 14)

### Сигнатуры, повторы и салит

Удаляем одинаковые пары запрос-ответ, формируем выборку из полноценных сигнатур и разделяем ее на тренировочную, валидационную и тестовую

In [33]:
# Отбрасываем повторы пар запрос-ответ
positive_pairs = validation_source.drop_duplicates(
    QUERY_COLS + ["item_id"]
).copy()
positive_pairs["query_group"] = (
    positive_pairs["search_query"].map(lambda text: " ".join(text.lower().split()))
)

query_groups = (
    pd.Series(sorted(positive_pairs["query_group"].unique()))
    .sample(frac=1, random_state=SPLIT_SEED)
    .tolist()
)
n_train = int(len(query_groups) * 0.70)
n_val = int(len(query_groups) * 0.15)
group_to_split = {
    text: (
        "train" if position < n_train
        else "val" if position < n_train + n_val
        else "test"
    )
    for position, text in enumerate(query_groups)
}
positive_pairs["split"] = positive_pairs["query_group"].map(group_to_split)

# Второй разрез делит полные сигнатуры, поэтому одинаковый текст может
# встретиться и в fit, и в holdout — как в benchmark.
signature_catalog = positive_pairs[QUERY_COLS].drop_duplicates().reset_index(drop=True)
signature_order = signature_catalog.sample(
    frac=1, random_state=SPLIT_SEED + 1
).index
n_signature_train = int(len(signature_catalog) * 0.70)
n_signature_val = int(len(signature_catalog) * 0.15)
signature_catalog["signature_split"] = "test"
signature_catalog.loc[signature_order[:n_signature_train], "signature_split"] = "train"
signature_catalog.loc[
    signature_order[n_signature_train:n_signature_train + n_signature_val],
    "signature_split",
] = "val"
positive_pairs = positive_pairs.merge(
    signature_catalog, on=QUERY_COLS, how="left", validate="many_to_one"
)

# Сортируем уникальные сигнатуры и назначаем локальные ID.
query_catalog = (
    positive_pairs[QUERY_COLS + ["query_group", "split", "signature_split"]]
    .drop_duplicates()
    .sort_values(QUERY_COLS, kind="stable")
    .reset_index(drop=True)
)
query_catalog.insert(
    0, "validation_query_id",
    [f"val_{i:07d}" for i in range(len(query_catalog))],
)
positive_pairs = positive_pairs.merge(
    query_catalog[QUERY_COLS + ["validation_query_id"]],
    on=QUERY_COLS, how="left", validate="many_to_one",
)

# Полная разметка, общая для двух протоколов.
full_relevance = (
    positive_pairs.groupby("validation_query_id")["item_id"].agg(frozenset)
)
query_input_cols = ["validation_query_id"] + QUERY_COLS
train_pairs = positive_pairs.loc[
    positive_pairs["split"].eq("train"),
    query_input_cols + ["item_id"],
].reset_index(drop=True)
train_item_ids = frozenset(train_pairs["item_id"])

split_groups = {
    split: set(query_catalog.loc[query_catalog["split"].eq(split), "query_group"])
    for split in SPLITS
}
assert query_catalog[QUERY_COLS].duplicated().sum() == 0
assert query_catalog["signature_split"].isin(SPLITS).all()

print(f"Уникальных пар: {len(positive_pairs):,}")


Уникальных пар: 467,054


### Общий конструктор выборок

Функция ниже пересекает полную разметку с выбранным корпусом и возвращает один DataFrame. Одна строка — одна полная сигнатура запроса, а колонка relevant_items содержит множество правильных item_id, доступных в этом корпусе. Запросы с пустым пересечением исключаются.

При поиске модели передаются только query_input_cols. Колонку relevant_items используем исключительно при расчёте метрики.


In [34]:
def build_validation_dataframe(corpus):
    """Собрать один DataFrame запросов и ответов для заданного корпуса."""
    allowed_item_ids = set(corpus["item_id"])
    result = query_catalog.copy()
    result["relevant_items"] = result["validation_query_id"].map(
        full_relevance.map(lambda item_ids: item_ids & allowed_item_ids)
    )
    result["n_relevant"] = result["relevant_items"].map(len)
    return result.loc[result["n_relevant"].gt(0)].reset_index(drop=True)


### 1. Объединённый корпус — основной протокол

Все train-позитивы есть в индексе, поэтому ни запросы, ни позитивы не исключаются. Все объявления остаются кандидатами и для train, и для val, и для test. Исключать validation-объявления из индекса нельзя: алгоритм должен иметь возможность их найти.


In [35]:
combined_validation = build_validation_dataframe(combined_corpus)
combined_train = combined_validation.loc[combined_validation["split"].eq("train")].reset_index(drop=True)
combined_val = combined_validation.loc[combined_validation["split"].eq("val")].reset_index(drop=True)
combined_test = combined_validation.loc[combined_validation["split"].eq("test")].reset_index(drop=True)

In [7]:
combined_validation.head()

,validation_query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,query_group,split,relevant_items,n_relevant
0,val_0000000,"""промывка радиатора автомобиля без снятия любе...",637640,0,,114,"""промывка радиатора автомобиля без снятия любе...",train,frozenset({791cc82871ec82d7}),1
1,val_0000001,"""ремонт ванной под ключ""",637640,0,Тип услуги Ремонт квартир и домов под ключ Вид...,114,"""ремонт ванной под ключ""",train,frozenset({87d23913a737eceb}),1
2,val_0000002,"""установка рулевой колонки на лодку пвх""",107620,0,,114,"""установка рулевой колонки на лодку пвх""",test,frozenset({9d22e6e798d0ef38}),1
3,val_0000003,'ktrnhbrf ghjdjlrf,650400,0,Тип услуги Электрика Кто оказывает услуги Част...,114,'ktrnhbrf ghjdjlrf,train,"frozenset({e9302367ecb66853, 74f4037ad0eaf7ce})",2
4,val_0000004,0445110298,638400,0,Техника Коммунальная техника Тип коммунальной ...,114,0445110298,train,frozenset({ad6a9348096a1213}),1


### 2. Корпус benchmark — протокол по пересечению позитивов

Поиск выполняется по всем benchmark_items, включая объявления без train-взаимодействий. Ground truth для каждого запроса — только его известные выбранные объявления из этого корпуса. Нулевое пересечение исключает запрос из DataFrame. Это условная метрика по доступным позитивам.


In [36]:
benchmark_validation = build_validation_dataframe(benchmark_corpus)
benchmark_train = benchmark_validation.loc[benchmark_validation["split"].eq("train")].reset_index(drop=True)
benchmark_val = benchmark_validation.loc[benchmark_validation["split"].eq("val")].reset_index(drop=True)
benchmark_test = benchmark_validation.loc[benchmark_validation["split"].eq("test")].reset_index(drop=True)

benchmark_signature_train = benchmark_validation.loc[benchmark_validation["signature_split"].eq("train")].reset_index(drop=True)
benchmark_signature_val = benchmark_validation.loc[benchmark_validation["signature_split"].eq("val")].reset_index(drop=True)
benchmark_signature_test = benchmark_validation.loc[benchmark_validation["signature_split"].eq("test")].reset_index(drop=True)

validation_summary = pd.DataFrame([
    {
        "protocol": name,
        "split": split,
        "queries": len(part),
        "positive_pairs": int(part["n_relevant"].sum()),
        "corpus_items": len(corpus),
    }
    for name, frame, corpus in [
        ("combined", combined_validation, combined_corpus),
        ("benchmark", benchmark_validation, benchmark_corpus),
    ]
    for split in SPLITS
    for part in [frame.loc[frame["split"].eq(split)]]
])
validation_summary

# Сверяем, насколько validation похожа на benchmark по знакомости текста.
normalize_query_group = lambda values: values.fillna("").str.lower().str.split().str.join(" ")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_texts = set(normalize_query_group(validation_source["search_query"]))
actual_benchmark_seen_text = normalize_query_group(benchmark_queries["search_query"]).isin(benchmark_texts).mean()
text_split_seen = normalize_query_group(benchmark_val["search_query"]).isin(
    set(normalize_query_group(benchmark_train["search_query"]))
).mean()
signature_split_seen = normalize_query_group(benchmark_signature_val["search_query"]).isin(
    set(normalize_query_group(benchmark_signature_train["search_query"]))
).mean()
print(f"Benchmark: {actual_benchmark_seen_text:.1%} известных текстов; text-split val: {text_split_seen:.1%}; signature-split val: {signature_split_seen:.1%}")

pd.concat([
    benchmark_val.assign(validation="text-disjoint val"),
    benchmark_test.assign(validation="text-disjoint holdout"),
    benchmark_signature_val.assign(validation="signature-split val"),
    benchmark_signature_test.assign(validation="signature-split holdout"),
]).groupby(["validation", "search_category"]).agg(
    queries=("validation_query_id", "size"),
    positive_pairs=("n_relevant", "sum"),
).reset_index()

,protocol,split,queries,positive_pairs,corpus_items
0,combined,train,246850,322952,515895
1,combined,val,53403,74389,515895
2,combined,test,54210,69713,515895
3,benchmark,train,18586,20667,189212
4,benchmark,val,4065,4597,189212
5,benchmark,test,3905,4308,189212


In [37]:
import re
import numpy as np
import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer


# Удаляем только подписи полей в параметрах, сохраняя значения.
# Длинные подписи проверяем раньше коротких: Тип услуги автосервиса / Тип услуги.
PARAM_LABELS = [
    'Вид услуги', 'Тип услуги автосервиса', 'Тип услуги',
    'Название услуги', 'Специальность или сфера',
    'Предмет или специальность', 'Место оказания услуг',
    'Тип стоимости', 'Начальная цена',
]
PARAM_LABEL_PATTERN = r'(?<!\w)(?:' + '|'.join(
    re.escape(label) for label in sorted(PARAM_LABELS, key=len, reverse=True)
) + r')(?!\w)'


# Переключатели для сравнения вариантов на одной валидации.
REMOVE_TECH_PARAMS = True
REMOVE_PROMO = True

# Удаляем только известные поля и их значения, не захватывая соседние поля.
DAY_VALUE = (r'(?i:понедельник|вторник|среда|четверг|пятница|суббота|воскресенье|'
             r'пн\.?|вт\.?|ср\.?|чт\.?|пт\.?|сб\.?|вс\.?)(?!\w)')
TECH_PARAM_PATTERNS = [
    r'График работы\s+(?:от|до)(?:\s+\d+(?::\d{2})?(?![\w:]))?',
    r'Время работы,?\s+(?:с|до)(?:\s+\d{1,2}:\d{2}(?![\w:]))?',
    r'(?:График работы,?\s+дни недели|Рабочие дни)(?:\s+' + DAY_VALUE + r')?',
    r'Признак предзаполнения прайс[ -]листа(?:\s+\d+(?!\w))?',
]
TECH_PARAM_PATTERN = r'(?<!\w)(?:' + '|'.join(TECH_PARAM_PATTERNS) + r')(?!\w)'

PROMO_PHRASES = [
    'индивидуальный подход', 'низкие цены',
    'добавляйте в избранное', 'добавьте в избранное',
    'звоните', 'пишите',
]
PROMO_PATTERN = r'(?i)(?<!\w)(?:' + '|'.join(
    re.escape(phrase) for phrase in PROMO_PHRASES
) + r')(?!\w)'


def clean_params(series):
    series = series.fillna('').str.replace(r'\s+', ' ', regex=True)
    if REMOVE_TECH_PARAMS:
        series = series.str.replace(TECH_PARAM_PATTERN, ' ', regex=True)
    # Регистр подписей сохраняем: значения со словами «вид» и «услуги» не удаляем.
    return (series.fillna('')
            .str.replace(PARAM_LABEL_PATTERN, ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True).str.strip())


def clean_description(series):
    series = series.fillna('').str.replace(r'\s+', ' ', regex=True)
    if REMOVE_PROMO:
        series = series.str.replace(PROMO_PATTERN, ' ', regex=True)
    return series.str.replace(r'\s+', ' ', regex=True).str.strip()


def normalize_text(series):
    # Одинаковая нормализация запросов и объявлений после очистки подписей.
    # Сохраняем цифры, латиницу, адреса, предлоги и отрицания.
    return (series.fillna('').str.lower()
            .str.replace('ё', 'е', regex=False)
            .str.replace(r'\s+', ' ', regex=True).str.strip())


def prepare_item_text(df):
	title = df["item_title_raw"].fillna("")
	params = clean_params(df["item_infm_params_text"])
	description = clean_description(df["item_description_raw"])

	# Заголовок повторяем, чтобы увеличить его влияние.
	return normalize_text(title + " " + title + " " + params + " " + description)


def prepare_query_text(df):
	query = df["search_query"].fillna("")
	filters = clean_params(df["search_infm_params_text"])

	return normalize_text(query + " " + filters)

vectorizer = TfidfVectorizer(
      lowercase=True,
      ngram_range=(1, 2),
      min_df=2,
      max_df=0.98,
      max_features=200_000,
      sublinear_tf=True,
      dtype=np.float32,
  )

item_texts = prepare_item_text(benchmark_corpus)
item_matrix = vectorizer.fit_transform(tqdm.tqdm(
          item_texts,
          total=len(item_texts),
          desc="TF-IDF корпуса",
      )
)

query_texts = prepare_query_text(benchmark_val)
query_matrix = vectorizer.transform(query_texts)

TF-IDF корпуса: 100%|██████████| 189212/189212 [01:38<00:00, 1928.64it/s]


In [38]:
from concurrent.futures import ThreadPoolExecutor

def get_predictions_sparse(
	query_df,
	items_df,
	query_matrix,
	item_matrix,
	batch_size=32,
	n_workers=8,
	top_k=50,
):
	item_ids = items_df["item_id"].to_numpy()
	query_ids = query_df["validation_query_id"].to_numpy()

	top_k = min(top_k, len(items_df))
	item_matrix_t = item_matrix.T.tocsc()
	predictions = {}

	batch_starts = list(range(0, len(query_df), batch_size))

	def process_batch(start):
		end = min(start + batch_size, len(query_df))

		scores = (
			query_matrix[start:end] @ item_matrix_t
		).toarray()

		top_indices = np.argpartition(
			scores,
			kth=scores.shape[1] - top_k,
			axis=1,
		)[:, -top_k:]

		top_scores = np.take_along_axis(
			scores,
			top_indices,
			axis=1,
		)
		order = np.argsort(top_scores, axis=1)[:, ::-1]

		top_indices = np.take_along_axis(
			top_indices,
			order,
			axis=1,
		)

		return start, end, top_indices

	with ThreadPoolExecutor(max_workers=n_workers) as executor:
		results = executor.map(process_batch, batch_starts)

		for start, end, top_indices in tqdm.tqdm(
			results,
			total=len(batch_starts),
			desc="Поиск кандидатов",
		):
			for local_idx, row_idx in enumerate(range(start, end)):
				predictions[query_ids[row_idx]] = (
					item_ids[top_indices[local_idx]].tolist()
				)

	return predictions

In [24]:
preds = get_predictions_sparse(benchmark_val, benchmark_corpus, query_matrix, item_matrix, batch_size=64)
preds

Поиск кандидатов:   0%|          | 0/64 [00:00<?, ?it/s]

Поиск кандидатов: 100%|██████████| 64/64 [00:18<00:00,  3.51it/s]


{'val_0000099': ['f5e12bf2a59c4d41',
  '52de15eee8f42142',
  '775ac4b325b0a4c5',
  '9391fbf022c87569',
  '9aae2b25aeb418f7',
  'a51b8f09d8e00216',
  'dd09a2bfab222d0f',
  '00fa4ae604d53623',
  '2fe419ca3b9b83a3',
  'ddf90b623eb2c3c0',
  '59f4e00691c1b733',
  'dac0dcd7d929e78e',
  '294e0c845e3f2bf4',
  '733d05dbbb3e37e3',
  '469d91ae66a2b77d',
  '4cac03b7af10d674',
  'e5cbab595e027f60',
  '5496b03fc45256c7',
  '38ccae3e34f8a53e',
  '756fb1ba2c59cdd3',
  'a017d41bf38a7bc6',
  'f69acbb660b66ea7',
  'b6170d267dab1781',
  '5d48e936e9024a73',
  'baa37630c3b4e644',
  '247122941148c9d9',
  'ccf3fdf5f5a6ee28',
  '6cb949d70341a638',
  '48a739a2da642dc5',
  '5f5c6470f2807449',
  '0f63555f7ab82b03',
  'bae0931702891ade',
  '5ab9ef733ccbb8cb',
  '82e72028d85b2ec3',
  '9cd114b1cfc02f21',
  'ca4772ac4f2290cf',
  '254e488a1d519152',
  'a0e0bd03b8e90aef',
  'a51534e08cc8b977',
  '4f1c13180bc46533',
  '9eb203f9e90036cf',
  'c9a823ae64fbd1eb',
  '5e3c7abf40f3dc8e',
  '767b1189792f06b2',
  'f87ee8f9a94792

### Оценка предсказаний

Функция принимает предсказания и DataFrame с колонками validation_query_id и relevant_items. Для каждого запроса считаем долю известных позитивов в корпусе, попавших в выдачу, затем усредняем по запросам. Не усредняем по строкам train: запрос с десятью позитивами должен иметь такой же вес, как запрос с одним.

Отсутствующий запрос — ошибка входа, а пустой список для существующего запроса даёт нулевой Recall. Дубликаты, больше 50 кандидатов и ID вне корпуса тоже считаем ошибкой, чтобы не скрывать проблемы генератора. Порядок кандидатов в пределах 50 не влияет на метрику.


In [39]:
def evaluate_recall_at_50(predictions, validation_df, allowed_item_ids):
    """Возвращает macro Recall@50 и таблицу результатов по запросам."""
    query_ids = set(validation_df["validation_query_id"])
    missing = query_ids - set(predictions)
    extra = set(predictions) - query_ids
    if missing or extra:
        raise ValueError(f"Пропущено запросов: {len(missing)}; лишних: {len(extra)}")

    allowed = frozenset(allowed_item_ids)
    rows = []
    for row in validation_df[["validation_query_id", "relevant_items"]].itertuples(index=False):
        query_id = row.validation_query_id
        relevant = row.relevant_items
        candidates = predictions[query_id]
        if isinstance(candidates, (str, bytes, Mapping)):
            raise TypeError(f"{query_id}: ожидается коллекция item_id, не строка")
        candidates = list(candidates)
        if len(candidates) > 50:
            raise ValueError(f"{query_id}: больше 50 кандидатов")
        candidate_set = set(candidates)
        if len(candidate_set) != len(candidates):
            raise ValueError(f"{query_id}: повторяющиеся кандидаты")
        if not candidate_set <= allowed:
            raise ValueError(f"{query_id}: item_id отсутствует в корпусе")
        hits = len(relevant & candidate_set)
        rows.append({
            "validation_query_id": query_id,
            "n_relevant": len(relevant),
            "n_candidates": len(candidates),
            "hits": hits,
            "recall_at_50": hits / len(relevant),
        })
    per_query = pd.DataFrame(rows)
    return per_query["recall_at_50"].mean(), per_query


In [40]:
recall_at_50, per_query = evaluate_recall_at_50(
	predictions=preds,
	validation_df=benchmark_val,
	allowed_item_ids=benchmark_item_ids,
)

print(f"Recall@50: {recall_at_50:.5f}")

Recall@50: 0.23053


In [61]:
from sklearn.feature_extraction.text import CountVectorizer

bm25_vectorizer = CountVectorizer(
	lowercase=True,
	ngram_range=(1, 1),
	min_df=2,
	max_df=0.98,
	max_features=200_000,
	dtype=np.float32,
)

benchmark_item_texts = prepare_item_text(benchmark_corpus)

item_count_matrix = bm25_vectorizer.fit_transform(
	tqdm.tqdm(
		benchmark_item_texts,
		total=len(benchmark_item_texts),
		desc="Построение BM25-корпуса",
	)
).tocsr()

def make_bm25_matrix(count_matrix, k1=1.5, b=0.75):
	count_matrix = count_matrix.tocsr().astype(np.float32)

	n_documents = count_matrix.shape[0]

	# Длина каждого объявления в токенах.
	document_lengths = np.asarray(
		count_matrix.sum(axis=1)
	).ravel()

	average_document_length = document_lengths.mean()

	# Сколько документов содержит каждое слово.
	document_frequency = np.asarray(
		(count_matrix > 0).sum(axis=0)
	).ravel()

	# Положительная версия IDF, используемая в современных реализациях BM25.
	idf = np.log(
		1
		+ (
			n_documents - document_frequency + 0.5
		)
		/ (
			document_frequency + 0.5
		)
	).astype(np.float32)

	bm25_matrix = count_matrix.copy()

	# Для каждого ненулевого элемента определяем его документ.
	row_indices = np.repeat(
		np.arange(n_documents),
		np.diff(bm25_matrix.indptr),
	)

	term_frequency = bm25_matrix.data

	length_normalization = (
		1
		- b
		+ b
		* document_lengths[row_indices]
		/ average_document_length
	)

	denominator = (
		term_frequency
		+ k1 * length_normalization
	)

	bm25_matrix.data = (
		idf[bm25_matrix.indices]
		* term_frequency
		* (k1 + 1)
		/ denominator
	).astype(np.float32)

	return bm25_matrix


def normalize_for_title_match(series):
	return (series.fillna('').str.lower()
		.str.replace('ё', 'е', regex=False)
		.str.replace(r'[^0-9a-zа-я]+', ' ', regex=True)
		.str.replace(r'\s+', ' ', regex=True).str.strip())


def build_title_index(items_df):
	"""Индекс token -> строки корпуса для быстрого поиска точной фразы."""
	normalized_titles = normalize_for_title_match(
		items_df['item_title_raw']
	).to_numpy()
	postings = {}

	for row_idx, title in enumerate(tqdm.tqdm(
		normalized_titles, desc='Индекс заголовков'
	)):
		for token in set(title.split()):
			postings.setdefault(token, []).append(row_idx)

	postings = {
		token: np.asarray(indices, dtype=np.int32)
		for token, indices in postings.items()
	}
	return normalized_titles, postings


def get_exact_title_predictions(
	query_df, items_df, query_matrix, item_matrix, top_k=50,
):
	"""Точные вхождения, ранжированные по локации, качеству и BM25."""
	normalized_titles, postings = build_title_index(items_df)
	normalized_queries = normalize_for_title_match(
		query_df['search_query']
	).to_numpy()
	item_ids = items_df['item_id'].to_numpy()
	item_locations = items_df['item_location_id'].to_numpy()
	ratings = pd.to_numeric(
		items_df['item_rating'], errors='coerce'
	).fillna(0).clip(lower=0).to_numpy(dtype=np.float32)
	review_counts = pd.to_numeric(
		items_df['item_rating_reviews_count'], errors='coerce'
	).fillna(0).clip(lower=0).to_numpy(dtype=np.float32)
	# Высокий рейтинг надёжнее, когда он подтверждён большим числом отзывов.
	quality_scores = ratings * np.log1p(review_counts)
	query_ids = query_df['validation_query_id'].to_numpy()
	query_locations = query_df['search_location_id'].to_numpy()
	predictions = {}

	for row_idx, (query_id, query) in enumerate(tqdm.tqdm(
		zip(query_ids, normalized_queries),
		total=len(query_df), desc='Точные совпадения заголовков',
	)):
		query_tokens = set(query.split())
		query_postings = [postings.get(token) for token in query_tokens]

		if not query or not query_postings or any(
			indices is None for indices in query_postings
		):
			predictions[query_id] = []
			continue

		# Проверять фразу достаточно среди объявлений с самым редким словом запроса.
		base_indices = min(query_postings, key=len)
		exact_indices = np.asarray([
			idx for idx in base_indices if query in normalized_titles[idx]
		], dtype=np.int32)

		if len(exact_indices) == 0:
			predictions[query_id] = []
			continue

		scores = (
			query_matrix[row_idx] @ item_matrix[exact_indices].T
		).toarray().ravel()
		same_location = (
			item_locations[exact_indices] == query_locations[row_idx]
		).astype(np.int8)
		# np.lexsort использует последний ключ как главный:
		# локация -> rating * log(1 + reviews) -> reviews -> BM25.
		order = np.lexsort((
			-scores,
			-review_counts[exact_indices],
			-quality_scores[exact_indices],
			-same_location,
		))[:top_k]
		predictions[query_id] = item_ids[exact_indices[order]].tolist()

	return predictions


def merge_exact_with_bm25(exact_predictions, bm25_predictions, exact_limit=20):
	"""Берём до exact_limit точных совпадений, остаток заполняем BM25."""
	merged = {}
	for query_id, bm25_items in bm25_predictions.items():
		items = list(exact_predictions[query_id][:exact_limit])
		seen = set(items)
		for item_id in bm25_items:
			if item_id not in seen:
				items.append(item_id)
				seen.add(item_id)
			if len(items) == 50:
				break
		merged[query_id] = items
	return merged


Построение BM25-корпуса: 100%|██████████| 189212/189212 [00:37<00:00, 5094.99it/s]


In [62]:
bm25_item_matrix = make_bm25_matrix(
	item_count_matrix,
	k1=1.5,
	b=0.75,
)

benchmark_query_texts = prepare_query_text(benchmark_val)

bm25_query_matrix = bm25_vectorizer.transform(
	benchmark_query_texts
).tocsr()

In [65]:
bm25_preds = get_predictions_sparse(
      query_df=benchmark_val,
      items_df=benchmark_corpus,
      query_matrix=bm25_query_matrix,
      item_matrix=bm25_item_matrix,
      batch_size=32,
      n_workers=8,
)

exact_title_preds = get_exact_title_predictions(
	query_df=benchmark_val,
	items_df=benchmark_corpus,
	query_matrix=bm25_query_matrix,
	item_matrix=bm25_item_matrix,
	top_k=50,
)

EXACT_TITLE_LIMIT = 25
bm25_exact_preds = merge_exact_with_bm25(
	exact_title_preds, bm25_preds, exact_limit=EXACT_TITLE_LIMIT,
)

Поиск кандидатов:   0%|          | 0/128 [00:00<?, ?it/s]

Точные совпадения заголовков: 100%|██████████| 4065/4065 [00:03<00:00, 1197.38it/s]


In [66]:
bm25_recall, bm25_per_query = evaluate_recall_at_50(
	predictions=bm25_preds,
	validation_df=benchmark_val,
	allowed_item_ids=benchmark_item_ids,
)

bm25_exact_recall, bm25_exact_per_query = evaluate_recall_at_50(
	predictions=bm25_exact_preds,
	validation_df=benchmark_val,
	allowed_item_ids=benchmark_item_ids,
)

queries_with_exact = sum(bool(items) for items in exact_title_preds.values())
print(f"BM25 Recall@50:              {bm25_recall:.5f}")
print(f"BM25 + exact title Recall@50: {bm25_exact_recall:.5f}")
print(f"Запросов с точными совпадениями: {queries_with_exact:,} / {len(benchmark_val):,}")

BM25 Recall@50:              0.28529
BM25 + exact title Recall@50: 0.43016
Запросов с точными совпадениями: 3,110 / 4,065


### Гибридный поиск: BM25 + Granite Embedding

Dense-канал использует локальную модель [ibm-granite/granite-embedding-97m-multilingual-r2](https://huggingface.co/ibm-granite/granite-embedding-97m-multilingual-r2) (Apache-2.0), [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html) и [FAISS](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes). Для воспроизведения нужны пакеты из `pyproject.toml`; при первом запуске веса скачиваются с Hugging Face, далее инференс выполняется локально (можно указать локальную папку модели вместо ID).

На Windows `uv sync --frozen` теперь устанавливает CUDA 13.0-сборку PyTorch из официального индекса. Ячейка проверяет `torch.cuda.is_available()` и останавливается с понятной ошибкой, если эмбеддер не может работать на GPU.

Используем только признаки запроса и объявления: разметка не участвует в поиске. В отличие от BM25, заголовок для эмбеддинга не повторяем. `EMBED_MAX_TOKENS` — явное ограничение входа модели по токенам ради памяти/скорости, а не обрезка текстовых колонок в данных. Кэш эмбеддингов объявлений зависит от текстов и этого лимита. FAISS HNSW даёт приблизительный dense top-k; полнота большого объединённого пула — **верхняя граница**, а итоговая метрика считается лишь для 50 объявлений.

Вес dense-канала RRF выбираем на `benchmark_signature_val`; результат на `benchmark_signature_test` считаем один раз после выбора. Это отдельный retrieval-эксперимент, он пока не переобучает логистический reranker ниже.

In [ ]:
import hashlib
import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "ibm-granite/granite-embedding-97m-multilingual-r2"
EMBED_MAX_TOKENS = 512  # Увеличить при достаточной VRAM; корпус тогда кодируется заново.
if not torch.cuda.is_available():
    raise RuntimeError("Нужна CUDA-сборка PyTorch и доступная NVIDIA GPU; проверьте uv sync и драйвер.")
EMBED_BATCH_SIZE = 16
EMBED_TOP_K = 200
EMBED_CACHE_DIR = DATA_DIR / "embedding_cache"
EMBED_CACHE_DIR.mkdir(exist_ok=True)

# Веса можно предварительно скачать и заменить EMBED_MODEL_ID на локальный путь.
embed_model = SentenceTransformer(
    EMBED_MODEL_ID,
    device="cuda",
)
assert embed_model.device.type == "cuda"
embed_model.max_seq_length = EMBED_MAX_TOKENS
print(f"Embedding device: {embed_model.device}; max tokens: {embed_model.max_seq_length}")

# Смысловые подписи оставляем, технические поля и рекламу чистим как для BM25.
dense_item_texts = (
    "Заголовок: " + benchmark_corpus["item_title_raw"].fillna("")
    + ". Параметры: " + clean_params(benchmark_corpus["item_infm_params_text"])
    + ". Описание: " + clean_description(benchmark_corpus["item_description_raw"])
).tolist()

# Хэш текстов защищает от использования кэша после изменения подготовки корпуса.
text_hashes = pd.util.hash_pandas_object(
    pd.Series(dense_item_texts), index=False,
).to_numpy(dtype=np.uint64)
corpus_fingerprint = hashlib.sha256(text_hashes.tobytes()).hexdigest()[:16]
embedding_cache_file = EMBED_CACHE_DIR / (
    f"granite_97m_r2_{EMBED_MAX_TOKENS}_{corpus_fingerprint}.npy"
)
if embedding_cache_file.exists():
    dense_item_embeddings = np.load(embedding_cache_file, mmap_mode="r")
    print(f"Эмбеддинги объявлений загружены из {embedding_cache_file}")
else:
    dense_item_embeddings = embed_model.encode_document(
        dense_item_texts,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(embedding_cache_file, dense_item_embeddings)
    print(f"Эмбеддинги объявлений сохранены в {embedding_cache_file}")
assert dense_item_embeddings.shape[0] == len(benchmark_corpus)
assert dense_item_embeddings.shape[1] == 384
print("Форма dense-матрицы:", dense_item_embeddings.shape)
del dense_item_texts, text_hashes  # После кэширования длинные строки больше не нужны.

In [ ]:
def prepare_dense_query_text(df):
    return (
        "Запрос: " + df["search_query"].fillna("")
        + ". Фильтры: " + clean_params(df["search_infm_params_text"])
    ).tolist()

# Нормализованные векторы + inner product дают cosine similarity.
dense_index = faiss.IndexHNSWFlat(
    dense_item_embeddings.shape[1], 32, faiss.METRIC_INNER_PRODUCT,
)
dense_index.hnsw.efConstruction = 160
dense_index.hnsw.efSearch = max(256, EMBED_TOP_K)
dense_index.add(np.ascontiguousarray(dense_item_embeddings, dtype=np.float32))
assert dense_index.ntotal == len(benchmark_corpus)


def get_dense_predictions(queries, top_k=EMBED_TOP_K, return_embeddings=False):
    query_embeddings = embed_model.encode_query(
        prepare_dense_query_text(queries),
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    _, item_rows = dense_index.search(
        np.ascontiguousarray(query_embeddings), top_k,
    )
    item_ids = benchmark_corpus["item_id"].to_numpy()
    predictions = {
        query_id: item_ids[rows[rows >= 0]].tolist()
        for query_id, rows in zip(queries["validation_query_id"], item_rows)
    }
    return (predictions, query_embeddings) if return_embeddings else predictions


def fuse_rrf(bm25_predictions, dense_predictions, dense_weight=1.0, k=60):
    """Reciprocal Rank Fusion: складываем ранговые вклады, без шкалирования scores."""
    fused = {}
    for query_id, bm25_items in bm25_predictions.items():
        scores = {}
        for rank, item_id in enumerate(bm25_items, 1):
            scores[item_id] = scores.get(item_id, 0.0) + 1.0 / (k + rank)
        for rank, item_id in enumerate(dense_predictions[query_id], 1):
            scores[item_id] = scores.get(item_id, 0.0) + dense_weight / (k + rank)
        fused[query_id] = sorted(scores, key=lambda item_id: (-scores[item_id], item_id))[:50]
    return fused


def retrieval_pool_recall(queries, *prediction_channels):
    """Верхняя граница Recall для объединения каналов до отбора 50."""
    recalls = []
    for row in queries[["validation_query_id", "relevant_items"]].itertuples(index=False):
        pool = set().union(*(channel[row.validation_query_id] for channel in prediction_channels))
        recalls.append(len(pool & row.relevant_items) / len(row.relevant_items))
    return float(np.mean(recalls))

In [ ]:
# BM25 и dense ищут по одному корпусу в одном и том же порядке item_id.
hybrid_val = benchmark_signature_val.reset_index(drop=True)
hybrid_test = benchmark_signature_test.reset_index(drop=True)


def get_bm25_top200(queries):
    query_matrix = bm25_vectorizer.transform(prepare_query_text(queries)).tocsr()
    return get_predictions_sparse(
        queries, benchmark_corpus, query_matrix, bm25_item_matrix,
        top_k=EMBED_TOP_K, batch_size=32, n_workers=2,
    )


bm25_val_200 = get_bm25_top200(hybrid_val)
bm25_test_200 = get_bm25_top200(hybrid_test)
dense_val_200, dense_val_embeddings = get_dense_predictions(hybrid_val, return_embeddings=True)
dense_test_200, dense_test_embeddings = get_dense_predictions(hybrid_test, return_embeddings=True)

# Сначала сравниваем отдельные каналы и достижимую полноту их объединения.
for name, queries, sparse, dense in [
    ("val", hybrid_val, bm25_val_200, dense_val_200),
    ("test", hybrid_test, bm25_test_200, dense_test_200),
]:
    bm25_top50 = {qid: items[:50] for qid, items in sparse.items()}
    dense_top50 = {qid: items[:50] for qid, items in dense.items()}
    bm25_recall, _ = evaluate_recall_at_50(bm25_top50, queries, benchmark_item_ids)
    dense_recall, _ = evaluate_recall_at_50(dense_top50, queries, benchmark_item_ids)
    pool_recall_value = retrieval_pool_recall(queries, sparse, dense)
    print(f"{name}: BM25@50={bm25_recall:.5f}; dense@50={dense_recall:.5f}; "
          f"union pool@200+200={pool_recall_value:.5f}")

# Настройка единственного веса по val; test не участвует в выборе.
RRF_DENSE_WEIGHTS = (0.25, 0.5, 1.0, 2.0, 4.0)
val_rrf_scores = {}
for weight in RRF_DENSE_WEIGHTS:
    predictions = fuse_rrf(bm25_val_200, dense_val_200, dense_weight=weight)
    val_rrf_scores[weight], _ = evaluate_recall_at_50(
        predictions, hybrid_val, benchmark_item_ids,
    )
    print(f"RRF dense_weight={weight:g}: val Recall@50={val_rrf_scores[weight]:.5f}")

# При равном Recall предпочитаем вес, ближайший к 1.
best_dense_weight = max(
    RRF_DENSE_WEIGHTS,
    key=lambda weight: (val_rrf_scores[weight], -abs(np.log2(weight))),
)
hybrid_val_preds = fuse_rrf(
    bm25_val_200, dense_val_200, dense_weight=best_dense_weight,
)
hybrid_test_preds = fuse_rrf(
    bm25_test_200, dense_test_200, dense_weight=best_dense_weight,
)
hybrid_test_recall, _ = evaluate_recall_at_50(
    hybrid_test_preds, hybrid_test, benchmark_item_ids,
)
print(f"Выбранный вес dense: {best_dense_weight:g}")
print(f"Hybrid Recall@50: val={val_rrf_scores[best_dense_weight]:.5f}; "
      f"test={hybrid_test_recall:.5f}")

### BM25 + exact-title → логистическая модель → top-50

Сначала выполняются предыдущие ячейки подготовки данных и BM25.
Каждый канал возвращает до 200 кандидатов. Логистическая модель обучается на
Основной reranker обучается на benchmark_signature_train, параметры проверяем на benchmark_signature_val.
Затем один раз измеряем результат на benchmark_signature_test. Этот split допускает
повтор текста запроса между train и holdout; text-disjoint split остаётся
отдельной проверкой для новых текстов.
Отсутствие объявления в известных положительных ответах считаем условным отрицательным примером.
Признаки: пересечение TF-IDF запроса и заголовка, BM25, exact-title, локация, рейтинг и отзывы.
Полнота пула — верхняя граница Recall@50 модели. На val ответы в пул не добавляем.


In [67]:
from collections import Counter, defaultdict
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

CANDIDATES_PER_CHANNEL = 200
NEGATIVES_PER_CHANNEL = 40
PAIR_BATCH_SIZE = 25_000
RERANK_SEED = 42

rerank_items = benchmark_corpus.reset_index(drop=True)
rerank_train_queries = benchmark_signature_train.reset_index(drop=True)
rerank_val_queries = benchmark_signature_val.reset_index(drop=True)
assert rerank_items["item_id"].is_unique
assert bm25_item_matrix.shape[0] == len(rerank_items)
# Матрица BM25 должна быть построена по benchmark_corpus в том же порядке.
assert set(rerank_train_queries["validation_query_id"]).isdisjoint(
    rerank_val_queries["validation_query_id"]
)
rerank_item_rows = pd.Series(
    np.arange(len(rerank_items)), index=rerank_items["item_id"]
)

# Словарь и IDF считаем по доступному корпусу, без меток релевантности.
pair_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1), min_df=2, max_features=200_000, dtype=np.float32,
)
rerank_titles = normalize_for_title_match(rerank_items["item_title_raw"])
pair_item_matrix = pair_vectorizer.fit_transform(rerank_titles).tocsr()
rerank_title_tokens = [set(text.split()) for text in rerank_titles]
rerank_title_array = rerank_titles.to_numpy()

rerank_locations = pd.to_numeric(
    rerank_items["item_location_id"], errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)
rerank_ratings = pd.to_numeric(
    rerank_items["item_rating"], errors="coerce"
).fillna(0).clip(0, 5).to_numpy(dtype=np.float32)
rerank_reviews = pd.to_numeric(
    rerank_items["item_rating_reviews_count"], errors="coerce"
).fillna(0).clip(lower=0).to_numpy(dtype=np.float32)


In [68]:
def get_topk_sparse_predictions(
    query_df, items_df, query_matrix, item_matrix,
    top_k, batch_size=32, n_workers=2,
):
    """Версия sparse-поиска с произвольным top_k для rerank-пула."""
    item_ids = items_df['item_id'].to_numpy()
    query_ids = query_df['validation_query_id'].to_numpy()
    top_k = min(top_k, len(items_df))
    item_matrix_t = item_matrix.T.tocsc()
    batch_starts = list(range(0, len(query_df), batch_size))

    def process_batch(start):
        end = min(start + batch_size, len(query_df))
        scores = (query_matrix[start:end] @ item_matrix_t).toarray()
        indices = np.argpartition(
            scores, kth=scores.shape[1] - top_k, axis=1,
        )[:, -top_k:]
        candidate_scores = np.take_along_axis(scores, indices, axis=1)
        order = np.argsort(candidate_scores, axis=1)[:, ::-1]
        return start, end, np.take_along_axis(indices, order, axis=1)

    predictions = {}
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        results = executor.map(process_batch, batch_starts)
        for start, end, indices in tqdm.tqdm(
            results, total=len(batch_starts), desc='BM25 top-k для reranker',
        ):
            for local_row, query_row in enumerate(range(start, end)):
                predictions[query_ids[query_row]] = (
                    item_ids[indices[local_row]].tolist()
                )
    return predictions


def build_rerank_pool(queries):
    """Естественный пул кандидатов: функция не читает relevant_items."""
    bm25_queries = bm25_vectorizer.transform(prepare_query_text(queries)).tocsr()
    bm25_candidates = get_topk_sparse_predictions(
        queries, rerank_items, bm25_queries, bm25_item_matrix,
        top_k=CANDIDATES_PER_CHANNEL, batch_size=32, n_workers=2,
    )
    # Сохраняем текущую эвристику exact-канала: локация, рейтинг/отзывы, BM25.
    exact_candidates = get_exact_title_predictions(
        queries, rerank_items, bm25_queries, bm25_item_matrix,
        top_k=CANDIDATES_PER_CHANNEL,
    )
    rows = []
    for query_row, query_id in enumerate(queries["validation_query_id"]):
        bm_ranks = {item: rank for rank, item in enumerate(bm25_candidates[query_id], 1)}
        ex_ranks = {item: rank for rank, item in enumerate(exact_candidates[query_id], 1)}
        for item in dict.fromkeys([*bm_ranks, *ex_ranks]):
            rows.append((
                query_row, rerank_item_rows[item],
                bm_ranks.get(item, 0), ex_ranks.get(item, 0),
            ))
    pool = pd.DataFrame(
        rows, columns=["query_row", "item_row", "bm25_rank", "exact_rank"],
        dtype=np.int32,
    )
    return pool, bm25_queries


def label_rerank_pairs(pairs, queries):
    """Метки используем при обучении или аудите; в признаки они не входят."""
    relevant = queries["relevant_items"].tolist()
    item_ids = rerank_items["item_id"].to_numpy()
    return np.fromiter(
        (item_ids[i] in relevant[q] for q, i in
         pairs[["query_row", "item_row"]].itertuples(index=False, name=None)),
        dtype=np.int8, count=len(pairs),
    )


def pool_recall(pairs, queries):
    hits = pd.Series(label_rerank_pairs(pairs, queries)).groupby(
        pairs["query_row"].to_numpy()
    ).sum().reindex(range(len(queries)), fill_value=0)
    return (hits.to_numpy() / queries["relevant_items"].map(len).to_numpy()).mean()


def build_query_history(train_queries):
    """Считает, какие объявления выбирали для каждого текста запроса в train."""
    history = defaultdict(Counter)
    for text, relevant in zip(train_queries["search_query"], train_queries["relevant_items"]):
        key = " ".join(text.lower().split()) if isinstance(text, str) else ""
        history[key].update(relevant)
    return history


def get_history_predictions(queries, history, limit=200):
    predictions = {}
    for query_id, text in zip(queries["validation_query_id"], queries["search_query"]):
        key = " ".join(text.lower().split()) if isinstance(text, str) else ""
        predictions[query_id] = [item for item, _ in history[key].most_common(limit)]
    return predictions


def merge_history_predictions(base, history_predictions, history_limit=20):
    """Добавляет top истории, затем заполняет список текущим reranker top-50."""
    merged = {}
    for query_id, base_items in base.items():
        items = list(dict.fromkeys(history_predictions.get(query_id, [])[:history_limit]))
        seen = set(items)
        for item_id in base_items:
            if item_id not in seen:
                items.append(item_id)
                seen.add(item_id)
            if len(items) == 50:
                break
        merged[query_id] = items
    return merged


def add_history_to_pool(pool, queries, history_predictions, limit=200):
    existing = set(zip(pool["query_row"], pool["item_row"]))
    rows = []
    for query_row, query_id in enumerate(queries["validation_query_id"]):
        for item_id in history_predictions.get(query_id, [])[:limit]:
            item_row = int(rerank_item_rows[item_id])
            if (query_row, item_row) not in existing:
                rows.append((query_row, item_row, 0, 0))
                existing.add((query_row, item_row))
    if rows:
        extra = pd.DataFrame(rows, columns=pool.columns).astype(np.int32)
        pool = pd.concat([pool, extra], ignore_index=True)
    return pool


In [ ]:
train_candidate_pool, rerank_train_bm25 = build_rerank_pool(rerank_train_queries)
val_candidate_pool, rerank_val_bm25 = build_rerank_pool(rerank_val_queries)

print(f"Пар в train-пуле: {len(train_candidate_pool):,}")
print(f"Пар в val-пуле: {len(val_candidate_pool):,}")
print(f"Train pool recall: {pool_recall(train_candidate_pool, rerank_train_queries):.5f}")
val_pool_recall = pool_recall(val_candidate_pool, rerank_val_queries)
print(f"Val pool recall (верхняя граница): {val_pool_recall:.5f}")


TypeError: get_predictions_sparse() got an unexpected keyword argument 'top_k'

In [ ]:
# Берём найденные положительные и верхние условно отрицательные из каждого канала.
train_targets = label_rerank_pairs(train_candidate_pool, rerank_train_queries)
keep = (
    (train_targets == 1)
    | train_candidate_pool["bm25_rank"].between(1, NEGATIVES_PER_CHANNEL)
    | train_candidate_pool["exact_rank"].between(1, NEGATIVES_PER_CHANNEL)
)
train_pair_samples = train_candidate_pool.loc[keep].copy()

# Только для train добавляем положительные, пропущенные обоими каналами.
# Это помогает учиться на всех известных положительных примерах.
positive_rows = [
    (query_row, int(rerank_item_rows[item_id]), 0, 0)
    for query_row, relevant in enumerate(rerank_train_queries["relevant_items"])
    for item_id in sorted(relevant)
]
train_pair_samples = (
    pd.concat([
        train_pair_samples,
        pd.DataFrame(positive_rows, columns=train_candidate_pool.columns),
    ], ignore_index=True)
    .drop_duplicates(["query_row", "item_row"])
    .reset_index(drop=True)
    .astype(np.int32)
)
train_y = label_rerank_pairs(train_pair_samples, rerank_train_queries)
print(f"Обучающих пар: {len(train_y):,}; положительных: {train_y.sum():,}")


In [ ]:
NUMERIC_FEATURES = [
    "log_bm25", "exact_title", "same_location", "rating",
    "log_reviews", "rating_x_log_reviews", "query_word_coverage", "title_cosine",
]


def make_pair_features(pairs, queries, query_tfidf, query_bm25):
    """Признаки пары. На вход не требуется ни одной метки релевантности."""
    q = pairs["query_row"].to_numpy()
    i = pairs["item_row"].to_numpy()
    # Один столбец на слово: произведение его TF-IDF весов в запросе и title.
    overlap = query_tfidf[q].multiply(pair_item_matrix[i]).tocsr()
    bm25_scores = np.asarray(
        query_bm25[q].multiply(bm25_item_matrix[i]).sum(axis=1)
    ).ravel()
    texts = normalize_for_title_match(queries["search_query"]).to_numpy()
    tokens = [set(text.split()) for text in texts]
    exact = np.fromiter(
        (bool(texts[a]) and f" {texts[a]} " in f" {rerank_title_array[b]} "
         for a, b in zip(q, i)), dtype=np.float32, count=len(pairs),
    )
    coverage = np.fromiter(
        (len(tokens[a] & rerank_title_tokens[b]) / max(1, len(tokens[a]))
         for a, b in zip(q, i)), dtype=np.float32, count=len(pairs),
    )
    query_locations = pd.to_numeric(
        queries["search_location_id"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)
    same_location = (
        np.isfinite(query_locations[q]) & np.isfinite(rerank_locations[i])
        & (query_locations[q] == rerank_locations[i])
    )
    log_reviews = np.log1p(rerank_reviews[i])
    numeric = np.column_stack([
        np.log1p(np.maximum(bm25_scores, 0)), exact, same_location,
        rerank_ratings[i], log_reviews, rerank_ratings[i] * log_reviews,
        coverage, np.asarray(overlap.sum(axis=1)).ravel(),
    ]).astype(np.float32)
    return overlap, numeric


rerank_train_tfidf = pair_vectorizer.transform(
    normalize_for_title_match(rerank_train_queries["search_query"])
).tocsr()
rerank_val_tfidf = pair_vectorizer.transform(
    normalize_for_title_match(rerank_val_queries["search_query"])
).tocsr()

# Порции ограничивают временную память при создании признаков.
overlap_parts, numeric_parts = [], []
for start in tqdm.tqdm(
    range(0, len(train_pair_samples), PAIR_BATCH_SIZE), desc="Train-признаки",
):
    overlap, numeric = make_pair_features(
        train_pair_samples.iloc[start:start + PAIR_BATCH_SIZE],
        rerank_train_queries, rerank_train_tfidf, rerank_train_bm25,
    )
    overlap_parts.append(overlap)
    numeric_parts.append(numeric)

train_overlap = vstack(overlap_parts, format="csr")
train_numeric = np.vstack(numeric_parts)
del overlap_parts, numeric_parts

# Масштабируем числовые признаки только по train. TF-IDF уже нормализован.
numeric_scaler = StandardScaler()
X_rerank_train = hstack([
    train_overlap,
    csr_matrix(numeric_scaler.fit_transform(train_numeric)),
], format="csr", dtype=np.float32)
del train_overlap, train_numeric
print("Матрица обучения:", X_rerank_train.shape)


In [ ]:
# Логистическая регрессия, обучаемая SGD: подходит для больших sparse-матриц.
# balanced — стартовая настройка, её и alpha можно сравнивать по val.
rerank_model = SGDClassifier(
    loss="log_loss", alpha=1e-5, class_weight="balanced",
    max_iter=30, tol=1e-3, random_state=RERANK_SEED,
)
rerank_model.fit(X_rerank_train, train_y)
print("Эпох обучения:", rerank_model.n_iter_)


In [ ]:
# На val оцениваем все естественно найденные пары, без добавления правильных ответов.
val_scores = np.empty(len(val_candidate_pool), dtype=np.float32)
for start in tqdm.tqdm(
    range(0, len(val_candidate_pool), PAIR_BATCH_SIZE), desc="Отбор кандидатов на val",
):
    end = min(start + PAIR_BATCH_SIZE, len(val_candidate_pool))
    overlap, numeric = make_pair_features(
        val_candidate_pool.iloc[start:end], rerank_val_queries,
        rerank_val_tfidf, rerank_val_bm25,
    )
    X_batch = hstack([
        overlap, csr_matrix(numeric_scaler.transform(numeric)),
    ], format="csr", dtype=np.float32)
    # Logit даёт тот же порядок, что predict_proba, без насыщения вероятностей.
    val_scores[start:end] = rerank_model.decision_function(X_batch)

ranked_val = val_candidate_pool.assign(score=val_scores).sort_values(
    ["query_row", "score", "item_row"],
    ascending=[True, False, True],
)
top50_val = ranked_val.groupby("query_row", sort=False).head(50).copy()
top50_val["item_id"] = rerank_items["item_id"].to_numpy()[top50_val["item_row"]]
items_by_query = top50_val.groupby("query_row")["item_id"].agg(list).to_dict()
rerank_preds = {
    query_id: items_by_query.get(row, [])
    for row, query_id in enumerate(rerank_val_queries["validation_query_id"])
}
rerank_recall, rerank_per_query = evaluate_recall_at_50(
    rerank_preds, rerank_val_queries, benchmark_item_ids,
)
print(f"Полнота пула:       {val_pool_recall:.5f}")
print(f"Логрег Recall@50:   {rerank_recall:.5f}")

# Baseline из тех же BM25-кандидатов, чтобы сравнение не зависело от старых переменных.
bm25_top50 = val_candidate_pool.loc[
    val_candidate_pool["bm25_rank"].between(1, 50)
].sort_values(["query_row", "bm25_rank"]).copy()
bm25_top50["item_id"] = rerank_items["item_id"].to_numpy()[bm25_top50["item_row"]]
baseline_by_row = bm25_top50.groupby("query_row")["item_id"].agg(list).to_dict()
baseline_preds = {
    query_id: baseline_by_row.get(row, [])
    for row, query_id in enumerate(rerank_val_queries["validation_query_id"])
}
baseline_recall, _ = evaluate_recall_at_50(
    baseline_preds, rerank_val_queries, benchmark_item_ids,
)
print(f"BM25 Recall@50:     {baseline_recall:.5f}")
print(f"Прирост к BM25:     {rerank_recall - baseline_recall:+.5f}")

# Запросная история доступна только при совпадающем тексте; метки val не используются.
query_history = build_query_history(rerank_train_queries)
val_history_predictions = get_history_predictions(
    rerank_val_queries, query_history, limit=CANDIDATES_PER_CHANNEL,
)
val_history_pool = add_history_to_pool(
    val_candidate_pool, rerank_val_queries, val_history_predictions,
)
val_pool_recall_with_history = pool_recall(val_history_pool, rerank_val_queries)
print(f"Полнота пула с историей: {val_pool_recall_with_history:.5f}")

for history_limit in (5, 10, 20):
    history_preds = merge_history_predictions(
        rerank_preds, val_history_predictions, history_limit=history_limit,
    )
    history_recall, _ = evaluate_recall_at_50(
        history_preds, rerank_val_queries, benchmark_item_ids,
    )
    print(f"Reranker + {history_limit} history items Recall@50: {history_recall:.5f}")

HISTORY_LIMIT = 20
val_history_preds = merge_history_predictions(
    rerank_preds, val_history_predictions, history_limit=HISTORY_LIMIT,
)
val_history_recall, _ = evaluate_recall_at_50(
    val_history_preds, rerank_val_queries, benchmark_item_ids,
)


In [ ]:
# Финальная проверка на signature-split holdout; HISTORY_LIMIT уже выбран по val.
rerank_holdout_queries = benchmark_signature_test.reset_index(drop=True)
holdout_candidate_pool, holdout_bm25 = build_rerank_pool(rerank_holdout_queries)
holdout_query_tfidf = pair_vectorizer.transform(
    normalize_for_title_match(rerank_holdout_queries["search_query"])
).tocsr()
holdout_scores = np.empty(len(holdout_candidate_pool), dtype=np.float32)
for start in tqdm.tqdm(
    range(0, len(holdout_candidate_pool), PAIR_BATCH_SIZE), desc="Holdout-признаки",
):
    end = min(start + PAIR_BATCH_SIZE, len(holdout_candidate_pool))
    overlap, numeric = make_pair_features(
        holdout_candidate_pool.iloc[start:end], rerank_holdout_queries,
        holdout_query_tfidf, holdout_bm25,
    )
    X_batch = hstack([
        overlap, csr_matrix(numeric_scaler.transform(numeric)),
    ], format="csr", dtype=np.float32)
    holdout_scores[start:end] = rerank_model.decision_function(X_batch)

ranked_holdout = holdout_candidate_pool.assign(score=holdout_scores).sort_values(
    ["query_row", "score", "item_row"], ascending=[True, False, True],
)
top50_holdout = ranked_holdout.groupby("query_row", sort=False).head(50).copy()
top50_holdout["item_id"] = rerank_items["item_id"].to_numpy()[top50_holdout["item_row"]]
holdout_by_row = top50_holdout.groupby("query_row")["item_id"].agg(list).to_dict()
holdout_preds = {
    query_id: holdout_by_row.get(row, [])
    for row, query_id in enumerate(rerank_holdout_queries["validation_query_id"])
}
holdout_recall, _ = evaluate_recall_at_50(
    holdout_preds, rerank_holdout_queries, benchmark_item_ids,
)
holdout_history_predictions = get_history_predictions(
    rerank_holdout_queries, query_history, limit=CANDIDATES_PER_CHANNEL,
)
holdout_history_pool = add_history_to_pool(
    holdout_candidate_pool, rerank_holdout_queries, holdout_history_predictions,
)
holdout_pool_recall = pool_recall(holdout_candidate_pool, rerank_holdout_queries)
holdout_pool_recall_with_history = pool_recall(holdout_history_pool, rerank_holdout_queries)
holdout_history_preds = merge_history_predictions(
    holdout_preds, holdout_history_predictions, history_limit=HISTORY_LIMIT,
)
holdout_history_recall, _ = evaluate_recall_at_50(
    holdout_history_preds, rerank_holdout_queries, benchmark_item_ids,
)
print(f"Holdout pool Recall: {holdout_pool_recall:.5f} → {holdout_pool_recall_with_history:.5f}")
print(f"Holdout reranker Recall@50: {holdout_recall:.5f}")
print(f"Holdout + {HISTORY_LIMIT} history items Recall@50: {holdout_history_recall:.5f}")

In [ ]:
def add_retrieval_predictions(pool, queries, predictions):
    existing = set(pool[["query_row", "item_row"]].itertuples(index=False, name=None))
    rows = []
    for query_row, query_id in enumerate(queries["validation_query_id"]):
        for item_id in predictions[query_id]:
            item_row = int(rerank_item_rows[item_id])
            if (query_row, item_row) not in existing:
                rows.append((query_row, item_row, 0, 0))
                existing.add((query_row, item_row))
    if rows:
        pool = pd.concat([
            pool, pd.DataFrame(rows, columns=pool.columns, dtype=np.int32),
        ], ignore_index=True)
    return pool


def score_candidate_pool(pool, queries, query_tfidf, query_bm25):
    scores = np.empty(len(pool), dtype=np.float32)
    for start in tqdm.tqdm(
        range(0, len(pool), PAIR_BATCH_SIZE), desc="TF-IDF pool scoring",
    ):
        end = min(start + PAIR_BATCH_SIZE, len(pool))
        overlap, numeric = make_pair_features(
            pool.iloc[start:end], queries, query_tfidf, query_bm25,
        )
        X_batch = hstack([
            overlap, csr_matrix(numeric_scaler.transform(numeric)),
        ], format="csr", dtype=np.float32)
        scores[start:end] = rerank_model.decision_function(X_batch)
    ranked = pool.assign(score=scores).sort_values(
        ["query_row", "score", "item_row"], ascending=[True, False, True],
    )
    top = ranked.groupby("query_row", sort=False).head(50).copy()
    top["item_id"] = rerank_items["item_id"].to_numpy()[top["item_row"]]
    by_row = top.groupby("query_row")["item_id"].agg(list).to_dict()
    return {qid: by_row.get(row, []) for row, qid in enumerate(queries["validation_query_id"])}


# Ещё один независимый канал: TF-IDF cosine по заголовкам объявлений.
val_title_query_matrix = pair_vectorizer.transform(
    normalize_for_title_match(rerank_val_queries["search_query"])
).tocsr()
val_title_candidates = get_topk_sparse_predictions(
    rerank_val_queries, rerank_items, val_title_query_matrix, pair_item_matrix,
    top_k=CANDIDATES_PER_CHANNEL, batch_size=32, n_workers=2,
)
val_title_pool = add_retrieval_predictions(
    val_candidate_pool, rerank_val_queries, val_title_candidates,
)
val_title_pool_recall = pool_recall(val_title_pool, rerank_val_queries)
val_title_preds = score_candidate_pool(
    val_title_pool, rerank_val_queries, rerank_val_tfidf, rerank_val_bm25,
)
val_title_recall, _ = evaluate_recall_at_50(
    val_title_preds, rerank_val_queries, benchmark_item_ids,
)
val_title_history_preds = merge_history_predictions(
    val_title_preds, val_history_predictions, history_limit=HISTORY_LIMIT,
)
val_title_history_recall, _ = evaluate_recall_at_50(
    val_title_history_preds, rerank_val_queries, benchmark_item_ids,
)

holdout_title_query_matrix = pair_vectorizer.transform(
    normalize_for_title_match(rerank_holdout_queries["search_query"])
).tocsr()
holdout_title_candidates = get_topk_sparse_predictions(
    rerank_holdout_queries, rerank_items, holdout_title_query_matrix, pair_item_matrix,
    top_k=CANDIDATES_PER_CHANNEL, batch_size=32, n_workers=2,
)
holdout_title_pool = add_retrieval_predictions(
    holdout_candidate_pool, rerank_holdout_queries, holdout_title_candidates,
)
holdout_title_pool_recall = pool_recall(holdout_title_pool, rerank_holdout_queries)
holdout_title_preds = score_candidate_pool(
    holdout_title_pool, rerank_holdout_queries, holdout_query_tfidf, holdout_bm25,
)
holdout_title_recall, _ = evaluate_recall_at_50(
    holdout_title_preds, rerank_holdout_queries, benchmark_item_ids,
)
holdout_title_history_preds = merge_history_predictions(
    holdout_title_preds, holdout_history_predictions, history_limit=HISTORY_LIMIT,
)
holdout_title_history_recall, _ = evaluate_recall_at_50(
    holdout_title_history_preds, rerank_holdout_queries, benchmark_item_ids,
)

print(f"Title TF-IDF pool Recall: val {val_pool_recall:.5f} → {val_title_pool_recall:.5f}; holdout {holdout_pool_recall:.5f} → {holdout_title_pool_recall:.5f}")
print(f"Reranker + title TF-IDF Recall@50: val {rerank_recall:.5f} → {val_title_recall:.5f}; holdout {holdout_recall:.5f} → {holdout_title_recall:.5f}")
print(f"+ history-{HISTORY_LIMIT}: val {val_history_recall:.5f} → {val_title_history_recall:.5f}; holdout {holdout_history_recall:.5f} → {holdout_title_history_recall:.5f}")

In [ ]:
# Вариант 2: обучаем reranker учитывать ранг из title-TF-IDF.
# В предыдущем тесте новый канал поднял полноту пула, но старый reranker
# не знал источник кандидата. Теперь добавляем title_rank как признак
# и берём hard negatives из этого канала.

def make_title_rank_frame(queries, predictions, top_k=None):
    query_ids = queries["validation_query_id"].to_numpy()
    candidate_lists = [
        predictions[query_id][:top_k] if top_k is not None else predictions[query_id]
        for query_id in query_ids
    ]
    sizes = np.fromiter((len(items) for items in candidate_lists), dtype=np.int32)
    total = int(sizes.sum())
    query_rows = np.repeat(np.arange(len(queries), dtype=np.int32), sizes)
    item_rows = np.fromiter(
        (int(rerank_item_rows[item_id])
         for items in candidate_lists for item_id in items),
        dtype=np.int32, count=total,
    )
    ranks = np.concatenate([
        np.arange(1, size + 1, dtype=np.int32) for size in sizes if size
    ])
    return pd.DataFrame({
        "query_row": query_rows,
        "item_row": item_rows,
        "title_rank": ranks,
    })


def merge_title_rank(pool, queries, predictions):
    title_pairs = make_title_rank_frame(queries, predictions)
    merged = pool.merge(
        title_pairs, on=["query_row", "item_row"],
        how="outer", sort=False, validate="one_to_one",
    )
    for column in ("bm25_rank", "exact_rank", "title_rank"):
        merged[column] = merged[column].fillna(0).astype(np.int32)
    return merged


# Для обучения добавляем тот же title-TF-IDF retrieval на train-запросах.
train_title_query_matrix = pair_vectorizer.transform(
    normalize_for_title_match(rerank_train_queries["search_query"])
).tocsr()
train_title_candidates = get_topk_sparse_predictions(
    rerank_train_queries, rerank_items, train_title_query_matrix, pair_item_matrix,
    top_k=CANDIDATES_PER_CHANNEL, batch_size=32, n_workers=2,
)

train_title_pool = merge_title_rank(
    train_candidate_pool, rerank_train_queries, train_title_candidates,
)
val_title_rank_pool = merge_title_rank(
    val_title_pool, rerank_val_queries, val_title_candidates,
)
holdout_title_rank_pool = merge_title_rank(
    holdout_title_pool, rerank_holdout_queries, holdout_title_candidates,
)
print(f"Title-channel train pool Recall: {pool_recall(train_title_pool, rerank_train_queries):.5f}")
print(f"Title-channel val/holdout pool Recall: "
      f"{pool_recall(val_title_rank_pool, rerank_val_queries):.5f} / "
      f"{pool_recall(holdout_title_rank_pool, rerank_holdout_queries):.5f}")


# Используем positives, найденные любым каналом, и hard negatives из top-40 каждого.
title_train_targets = label_rerank_pairs(train_title_pool, rerank_train_queries)
title_train_keep = (
    (title_train_targets == 1)
    | train_title_pool["bm25_rank"].between(1, NEGATIVES_PER_CHANNEL)
    | train_title_pool["exact_rank"].between(1, NEGATIVES_PER_CHANNEL)
    | train_title_pool["title_rank"].between(1, NEGATIVES_PER_CHANNEL)
)
title_train_samples = train_title_pool.loc[title_train_keep].copy()

title_positive_rows = [
    (query_row, int(rerank_item_rows[item_id]), 0, 0, 0)
    for query_row, relevant in enumerate(rerank_train_queries["relevant_items"])
    for item_id in sorted(relevant)
]
title_train_samples = (
    pd.concat([
        title_train_samples,
        pd.DataFrame(
            title_positive_rows,
            columns=train_title_pool.columns,
        ),
    ], ignore_index=True)
    .drop_duplicates(["query_row", "item_row"])
    .reset_index(drop=True)
    .astype(np.int32)
)
title_train_y = label_rerank_pairs(title_train_samples, rerank_train_queries)
print(f"Title-ranker training pairs: {len(title_train_y):,}; "
      f"positives: {title_train_y.sum():,}")


def make_title_rank_features(pairs, queries, query_tfidf, query_bm25):
    overlap, numeric = make_pair_features(
        pairs, queries, query_tfidf, query_bm25,
    )
    ranks = pairs["title_rank"].to_numpy(dtype=np.float32)
    title_rank_signal = np.zeros(len(ranks), dtype=np.float32)
    has_title_rank = ranks > 0
    title_rank_signal[has_title_rank] = 1.0 / np.log2(
        ranks[has_title_rank] + 1.0
    )
    numeric = np.column_stack([numeric, title_rank_signal]).astype(np.float32)
    return overlap, numeric


title_overlap_parts, title_numeric_parts = [], []
for start in tqdm.tqdm(
    range(0, len(title_train_samples), PAIR_BATCH_SIZE),
    desc="Title-ranker train features",
):
    end = min(start + PAIR_BATCH_SIZE, len(title_train_samples))
    overlap, numeric = make_title_rank_features(
        title_train_samples.iloc[start:end], rerank_train_queries,
        rerank_train_tfidf, rerank_train_bm25,
    )
    title_overlap_parts.append(overlap)
    title_numeric_parts.append(numeric)

title_train_overlap = vstack(title_overlap_parts, format="csr")
title_train_numeric = np.vstack(title_numeric_parts)
title_numeric_scaler = StandardScaler()
title_train_matrix = hstack([
    title_train_overlap,
    csr_matrix(title_numeric_scaler.fit_transform(title_train_numeric)),
], format="csr", dtype=np.float32)
del title_overlap_parts, title_numeric_parts, title_train_overlap, title_train_numeric

title_rank_model = SGDClassifier(
    loss="log_loss", alpha=1e-5, class_weight="balanced",
    max_iter=30, tol=1e-3, random_state=RERANK_SEED,
)
title_rank_model.fit(title_train_matrix, title_train_y)
print(f"Title-ranker epochs: {title_rank_model.n_iter_}")


def score_title_ranked_pool(pool, queries, query_tfidf, query_bm25):
    scores = np.empty(len(pool), dtype=np.float32)
    for start in tqdm.tqdm(
        range(0, len(pool), PAIR_BATCH_SIZE),
        desc="Title-ranker scoring",
    ):
        end = min(start + PAIR_BATCH_SIZE, len(pool))
        overlap, numeric = make_title_rank_features(
            pool.iloc[start:end], queries, query_tfidf, query_bm25,
        )
        matrix = hstack([
            overlap,
            csr_matrix(title_numeric_scaler.transform(numeric)),
        ], format="csr", dtype=np.float32)
        scores[start:end] = title_rank_model.decision_function(matrix)

    ranked = pool.assign(score=scores).sort_values(
        ["query_row", "score", "item_row"],
        ascending=[True, False, True],
    )
    top = ranked.groupby("query_row", sort=False).head(50).copy()
    top["item_id"] = rerank_items["item_id"].to_numpy()[top["item_row"]]
    by_row = top.groupby("query_row")["item_id"].agg(list).to_dict()
    return {
        query_id: by_row.get(row, [])
        for row, query_id in enumerate(queries["validation_query_id"])
    }


title_rank_val_preds = score_title_ranked_pool(
    val_title_rank_pool, rerank_val_queries,
    rerank_val_tfidf, rerank_val_bm25,
)
title_rank_val_recall, _ = evaluate_recall_at_50(
    title_rank_val_preds, rerank_val_queries, benchmark_item_ids,
)
title_rank_val_history_preds = merge_history_predictions(
    title_rank_val_preds, val_history_predictions, history_limit=HISTORY_LIMIT,
)
title_rank_val_history_recall, _ = evaluate_recall_at_50(
    title_rank_val_history_preds, rerank_val_queries, benchmark_item_ids,
)

title_rank_holdout_preds = score_title_ranked_pool(
    holdout_title_rank_pool, rerank_holdout_queries,
    holdout_query_tfidf, holdout_bm25,
)
title_rank_holdout_recall, _ = evaluate_recall_at_50(
    title_rank_holdout_preds, rerank_holdout_queries, benchmark_item_ids,
)
title_rank_holdout_history_preds = merge_history_predictions(
    title_rank_holdout_preds, holdout_history_predictions,
    history_limit=HISTORY_LIMIT,
)
title_rank_holdout_history_recall, _ = evaluate_recall_at_50(
    title_rank_holdout_history_preds, rerank_holdout_queries,
    benchmark_item_ids,
)

print(f"Title-rank reranker Recall@50: val {rerank_recall:.5f} -> "
      f"{title_rank_val_recall:.5f}; holdout {holdout_recall:.5f} -> "
      f"{title_rank_holdout_recall:.5f}")
print(f"With history-{HISTORY_LIMIT}: val {val_history_recall:.5f} -> "
      f"{title_rank_val_history_recall:.5f}; holdout "
      f"{holdout_history_recall:.5f} -> {title_rank_holdout_history_recall:.5f}")


### Dense-кандидаты в текущем reranker

Теперь dense top-200 добавляется к существующему BM25/exact/title-пулу. Для обучения логистической модели используем только `benchmark_signature_train`: сохраняем прежние обучающие пары и добавляем top-40 dense как hard negatives. Известные положительные пары, даже если retrieval их пропустил, остаются **только в обучении**. На val и test правильные ответы никогда не подмешиваются в кандидатов.

К уже имеющимся признакам добавляются `dense_rank` (сигнал 1/log₂(ранг+1)) и cosine similarity нормализованных эмбеддингов пары. Сравниваем с прежней моделью на том же корпусе, split и истории top-20; это измеряет добавочную пользу dense-канала для текущего решения, а не только качество отдельного RRF.

In [ ]:
assert list(hybrid_val["validation_query_id"]) == list(rerank_val_queries["validation_query_id"])
assert list(hybrid_test["validation_query_id"]) == list(rerank_holdout_queries["validation_query_id"])
assert bm25_item_matrix.shape[0] == dense_item_embeddings.shape[0] == len(rerank_items)

# Кодируем train-запросы один раз: те же векторы нужны для поиска и pair features.
dense_train_200, dense_train_embeddings = get_dense_predictions(
    rerank_train_queries, return_embeddings=True,
)


def merge_dense_rank(pool, queries, dense_predictions):
    dense_pairs = make_title_rank_frame(queries, dense_predictions).rename(
        columns={"title_rank": "dense_rank"}
    )
    merged = pool.merge(
        dense_pairs, on=["query_row", "item_row"],
        how="outer", sort=False, validate="one_to_one",
    )
    for column in ("bm25_rank", "exact_rank", "title_rank", "dense_rank"):
        merged[column] = merged[column].fillna(0).astype(np.int32)
    return merged


# Полный dense top-200 оставляем в val/test-пуле; для обучения достаточно
# первых 40 dense-кандидатов как новых сложных отрицательных примеров.
train_dense_rank_frame = make_title_rank_frame(
    rerank_train_queries, dense_train_200,
).rename(columns={"title_rank": "dense_rank"})
dense_hard_pairs = train_dense_rank_frame.loc[
    train_dense_rank_frame["dense_rank"].le(NEGATIVES_PER_CHANNEL),
    ["query_row", "item_row"],
]
train_hybrid_samples = pd.concat([
    title_train_samples,
    dense_hard_pairs.assign(
        bm25_rank=0, exact_rank=0, title_rank=0,
    )[title_train_samples.columns],
], ignore_index=True).drop_duplicates(["query_row", "item_row"])
train_hybrid_samples = train_hybrid_samples.merge(
    train_dense_rank_frame, on=["query_row", "item_row"],
    how="left", sort=False, validate="one_to_one",
)
train_hybrid_samples["dense_rank"] = (
    train_hybrid_samples["dense_rank"].fillna(0).astype(np.int32)
)
train_hybrid_samples = train_hybrid_samples.reset_index(drop=True)
train_hybrid_y = label_rerank_pairs(train_hybrid_samples, rerank_train_queries)
assert train_hybrid_y.sum() == title_train_y.sum()

val_hybrid_pool = merge_dense_rank(
    val_title_rank_pool, rerank_val_queries, dense_val_200,
)
holdout_hybrid_pool = merge_dense_rank(
    holdout_title_rank_pool, rerank_holdout_queries, dense_test_200,
)
val_hybrid_pool_recall = pool_recall(val_hybrid_pool, rerank_val_queries)
holdout_hybrid_pool_recall = pool_recall(holdout_hybrid_pool, rerank_holdout_queries)
print(f"Train pairs: {len(train_hybrid_samples):,}; positives: {train_hybrid_y.sum():,}")
print(f"Val pool Recall: {pool_recall(val_title_rank_pool, rerank_val_queries):.5f} → {val_hybrid_pool_recall:.5f}")
print(f"Holdout pool Recall: {pool_recall(holdout_title_rank_pool, rerank_holdout_queries):.5f} → {holdout_hybrid_pool_recall:.5f}")

In [ ]:
def make_hybrid_pair_features(pairs, queries, query_tfidf, query_bm25, query_embeddings):
    overlap, numeric = make_title_rank_features(
        pairs, queries, query_tfidf, query_bm25,
    )
    query_rows = pairs["query_row"].to_numpy()
    item_rows = pairs["item_row"].to_numpy()
    # Оба набора эмбеддингов L2-нормализованы, поэтому dot product = cosine.
    cosine = np.einsum(
        "ij,ij->i",
        np.asarray(query_embeddings[query_rows], dtype=np.float32),
        np.asarray(dense_item_embeddings[item_rows], dtype=np.float32),
    )
    ranks = pairs["dense_rank"].to_numpy(dtype=np.float32)
    rank_signal = np.zeros(len(ranks), dtype=np.float32)
    found = ranks > 0
    rank_signal[found] = 1.0 / np.log2(ranks[found] + 1.0)
    numeric = np.column_stack([numeric, cosine, rank_signal]).astype(np.float32)
    return overlap, numeric


hybrid_overlap_parts, hybrid_numeric_parts = [], []
for start in tqdm.tqdm(
    range(0, len(train_hybrid_samples), PAIR_BATCH_SIZE),
    desc="Hybrid train features",
):
    end = min(start + PAIR_BATCH_SIZE, len(train_hybrid_samples))
    overlap, numeric = make_hybrid_pair_features(
        train_hybrid_samples.iloc[start:end], rerank_train_queries,
        rerank_train_tfidf, rerank_train_bm25, dense_train_embeddings,
    )
    hybrid_overlap_parts.append(overlap)
    hybrid_numeric_parts.append(numeric)

hybrid_train_overlap = vstack(hybrid_overlap_parts, format="csr")
hybrid_train_numeric = np.vstack(hybrid_numeric_parts)
hybrid_numeric_scaler = StandardScaler()
hybrid_train_matrix = hstack([
    hybrid_train_overlap,
    csr_matrix(hybrid_numeric_scaler.fit_transform(hybrid_train_numeric)),
], format="csr", dtype=np.float32)
del hybrid_overlap_parts, hybrid_numeric_parts, hybrid_train_overlap, hybrid_train_numeric

hybrid_rank_model = SGDClassifier(
    loss="log_loss", alpha=1e-5, class_weight="balanced",
    max_iter=30, tol=1e-3, random_state=RERANK_SEED,
)
hybrid_rank_model.fit(hybrid_train_matrix, train_hybrid_y)
print("Hybrid train matrix:", hybrid_train_matrix.shape)
print("Epochs:", hybrid_rank_model.n_iter_)

In [ ]:
def score_hybrid_pool(pool, queries, query_tfidf, query_bm25, query_embeddings):
    scores = np.empty(len(pool), dtype=np.float32)
    for start in tqdm.tqdm(
        range(0, len(pool), PAIR_BATCH_SIZE), desc="Hybrid scoring",
    ):
        end = min(start + PAIR_BATCH_SIZE, len(pool))
        overlap, numeric = make_hybrid_pair_features(
            pool.iloc[start:end], queries, query_tfidf,
            query_bm25, query_embeddings,
        )
        matrix = hstack([
            overlap,
            csr_matrix(hybrid_numeric_scaler.transform(numeric)),
        ], format="csr", dtype=np.float32)
        scores[start:end] = hybrid_rank_model.decision_function(matrix)
    ranked = pool.assign(score=scores).sort_values(
        ["query_row", "score", "item_row"], ascending=[True, False, True],
    )
    top = ranked.groupby("query_row", sort=False).head(50).copy()
    top["item_id"] = rerank_items["item_id"].to_numpy()[top["item_row"]]
    by_row = top.groupby("query_row")["item_id"].agg(list).to_dict()
    return {
        query_id: by_row.get(row, [])
        for row, query_id in enumerate(queries["validation_query_id"])
    }


hybrid_rank_val_preds = score_hybrid_pool(
    val_hybrid_pool, rerank_val_queries,
    rerank_val_tfidf, rerank_val_bm25, dense_val_embeddings,
)
hybrid_rank_holdout_preds = score_hybrid_pool(
    holdout_hybrid_pool, rerank_holdout_queries,
    holdout_query_tfidf, holdout_bm25, dense_test_embeddings,
)
hybrid_rank_val_history_preds = merge_history_predictions(
    hybrid_rank_val_preds, val_history_predictions,
    history_limit=HISTORY_LIMIT,
)
hybrid_rank_holdout_history_preds = merge_history_predictions(
    hybrid_rank_holdout_preds, holdout_history_predictions,
    history_limit=HISTORY_LIMIT,
)
hybrid_rank_val_recall, hybrid_rank_val_per_query = evaluate_recall_at_50(
    hybrid_rank_val_history_preds, rerank_val_queries, benchmark_item_ids,
)
hybrid_rank_holdout_recall, hybrid_rank_holdout_per_query = evaluate_recall_at_50(
    hybrid_rank_holdout_history_preds, rerank_holdout_queries, benchmark_item_ids,
)

comparison = pd.DataFrame([
    {"split": "val", "title_rank_baseline": title_rank_val_history_recall,
     "with_dense": hybrid_rank_val_recall,
     "delta": hybrid_rank_val_recall - title_rank_val_history_recall,
     "pool_before": pool_recall(val_title_rank_pool, rerank_val_queries),
     "pool_after": val_hybrid_pool_recall},
    {"split": "holdout", "title_rank_baseline": title_rank_holdout_history_recall,
     "with_dense": hybrid_rank_holdout_recall,
     "delta": hybrid_rank_holdout_recall - title_rank_holdout_history_recall,
     "pool_before": pool_recall(holdout_title_rank_pool, rerank_holdout_queries),
     "pool_after": holdout_hybrid_pool_recall},
])
display(comparison)